# 05. Lựa chọn đặc trưng

Bộ B có nhiều biến đếm và thống kê lịch sử có thể chứa thông tin trùng nhau. Notebook dùng phương pháp Filter để loại cột hằng và giảm các biến số tương quan cao, không dùng TARGET.

Selector được fit riêng trên phần train của từng fold. Sau đó fit lại cùng quy tắc trên toàn train để xuất danh sách cho lần huấn luyện cuối. Kết quả giữ 181 biến/fold và 182 biến trên toàn train; hai danh sách phục vụ hai bước khác nhau.

Notebook 06 đã sử dụng quy trình này để hoàn tất 63 lượt CV. Phần dưới trình bày phương pháp và đầu ra chọn biến; kết quả so sánh mô hình được trình bày riêng ở notebook 06.



## Mục lục

- [01. Thiết lập](#part-1)
- [02. Đọc bộ đặc trưng train](#part-2)
- [03. Phương pháp Filter](#part-3)
- [04. Fit selector trong từng CV fold](#part-4)
- [05. Fit trên toàn train và xuất tập biến](#part-5)
- [06. Chốt kết quả và bàn giao bước tiếp theo](#part-6)

<a id="part-1"></a>

## 01. Thiết lập

Giữ nguyên split và ba CV fold đã có. Notebook chỉ đọc đặc trưng và chỉ mục train từ các file đã xuất ở bước trước; không đọc nhãn TARGET, validation hoặc test.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / 'reports/modeling/feature_selection_20261028'
OUTPUT.mkdir(parents=True, exist_ok=True)

CORRELATION_LIMIT = 0.90
MIN_PAIRWISE_OVERLAP = 0.50

**Kết luận:** giữ nguyên train và ba fold đã chốt; ngưỡng tương quan là 0,90 và tỷ lệ quan sát chung tối thiểu là 50%. Các tham số này được truyền trực tiếp vào selector.


<a id="part-2"></a>

## 02. Đọc bộ đặc trưng train

Bộ B gồm 199 biến: 183 biến số và 16 biến phân loại. SK_ID_CURR dùng làm index; TARGET, split và cv_fold không phải biến đầu vào.

In [2]:
schemas = json.loads(
    (ROOT / 'configs/preprocessing_schema.json').read_text(encoding='utf-8'))['schemas']
schema_b = schemas['B']
x = pd.read_parquet(ROOT / 'data/features/b/train.parquet').set_index('SK_ID_CURR')
x = x.loc[:, schema_b['feature_order']]
split = pd.read_parquet(
    ROOT / 'data/splits/application_split.parquet').set_index('SK_ID_CURR')

if not x.index.is_unique or not split.index.is_unique:
    raise ValueError('ID phải duy nhất trước khi chọn biến.')
train_split = split.loc[x.index]
if not train_split['split'].eq('train').all():
    raise ValueError('Chỉ được dùng hồ sơ train để fit selector.')
if {'TARGET', 'SK_ID_CURR', 'split', 'cv_fold'} & set(x.columns):
    raise ValueError('Nhãn hoặc khóa/phân tập không được thuộc đặc trưng.')
folds = train_split['cv_fold'].astype(int)
if set(folds.unique()) != {0, 1, 2}:
    raise ValueError('Dùng đúng ba fold train đã chốt.')

display(pd.DataFrame([{
    'train_rows': len(x),
    'input_features': len(x.columns),
    'numeric_features': len(schema_b['numeric']),
    'categorical_features': len(schema_b['categorical'])
}]))


,train_rows,input_features,numeric_features,categorical_features
0,184506,199,183,16


**Kết luận:** đầu vào gồm 184.506 hồ sơ train và 199 đặc trưng. TARGET, ID và thông tin phân tập được tách khỏi các biến dùng chọn lọc.


<a id="part-3"></a>

## 03. Phương pháp Filter

1. Bỏ cột hằng, tính cả trạng thái thiếu. Không đặt ngưỡng near-zero variance để loại cờ hiếm.
2. Tính Pearson giữa các biến số. Chỉ xét cặp có ít nhất 50% hồ sơ fit quan sát được cả hai giá trị.
3. Với |r| ≥ 0,90, ưu tiên biến ít thiếu hơn; nếu bằng nhau, giữ theo thứ tự schema.
4. Giữ biến phân loại trừ khi cột hằng trong phần fit. Đầu ra giữ thứ tự cột gốc.

Ngưỡng 0,90, mức quan sát chung 50% và quy tắc ưu tiên là thiết kế của nghiên cứu này, không phải quy định bắt buộc của giáo trình. Pearson chỉ đo quan hệ tuyến tính; bộ lọc chưa chứng minh rằng hiệu năng dự báo sẽ tăng.

In [3]:
from models.FeatureSelection import select_filter_features


**Kết luận:** bộ lọc ưu tiên biến ít thiếu hơn trong các nhóm trùng thông tin, đồng thời giữ cờ hiếm nếu cờ đó không hằng. Pearson chỉ kiểm tra quan hệ tuyến tính giữa các biến số.


<a id="part-4"></a>

## 04. Fit selector trong từng CV fold

Mỗi lượt dùng hai fold để học danh sách biến, rồi áp dụng danh sách ấy cho cả phần fit và phần giữ lại. Mọi thuật toán trong cùng một fold dùng chung danh sách. Phần giữ lại không tham gia tính tương quan hoặc độ thiếu để chọn biến.

Danh sách fit trên toàn train được dành cho mô hình cuối, không dùng để đánh giá ngược lại các CV fold.


In [4]:
selected_by_fold = {}
dropped_by_fold = []
fold_summary = []
for fold in [0, 1, 2]:
    fit_mask = folds.ne(fold)
    selected, dropped = select_filter_features(
        x.loc[fit_mask], schema_b, correlation_limit=CORRELATION_LIMIT,
        min_pairwise_overlap=MIN_PAIRWISE_OVERLAP)
    selected_by_fold[str(fold)] = selected
    dropped_by_fold.extend({'fold': fold, **row} for row in dropped)
    fold_summary.append({
        'fold': fold,
        'fit_rows': int(fit_mask.sum()),
        'probe_rows': int((~fit_mask).sum()),
        'input_features': len(x.columns),
        'selected_features': len(selected),
        'constant_removed': sum(row['reason'] == 'zero_variance' for row in dropped),
        'correlated_removed': sum(row['reason'] != 'zero_variance' for row in dropped)
    })

(OUTPUT / 'selected_features_by_fold.json').write_text(
    json.dumps(selected_by_fold, indent=2), encoding='utf-8')
pd.DataFrame(dropped_by_fold).to_csv(
    OUTPUT / 'dropped_features_by_fold.csv', index=False)
pd.DataFrame(fold_summary).to_csv(OUTPUT / 'selection_summary_by_fold.csv', index=False)
display(pd.DataFrame(fold_summary))
display(pd.DataFrame(dropped_by_fold))


,fold,fit_rows,probe_rows,input_features,selected_features,constant_removed,correlated_removed
0,0,123004,61502,199,181,1,17
1,1,123004,61502,199,181,1,17
2,2,123004,61502,199,181,1,17


,fold,feature,reason,kept_instead,absolute_correlation
0,0,FLAG_MOBIL,zero_variance,None,NaN
1,0,REGION_RATING_CLIENT_W_CITY,pearson_correlation_ge_0.90,REGION_RATING_CLIENT,0.951656
2,0,EMPLOYED_SENTINEL,pearson_correlation_ge_0.90,FLAG_EMP_PHONE,0.999835
3,0,BUR_CLOSED_N,pearson_correlation_ge_0.90,BUR_LOAN_N,0.932277
4,0,BB_CLOSED_MONTH_N,pearson_correlation_ge_0.90,BB_MONTH_N,0.908290
5,0,POS_LOAN_N,pearson_correlation_ge_0.90,PREV_APPROVED_N,0.952064
6,0,POS_KNOWN_DPD_MONTH_N,pearson_correlation_ge_0.90,POS_MONTH_N,1.000000
7,0,CC_KNOWN_DPD_MONTH_N,pearson_correlation_ge_0.90,CC_MONTH_N,1.000000
8,0,INS_OBS_VERSIONED_PERIOD_N,pearson_correlation_ge_0.90,INS_OBS_PAYMENT_ROW_N,0.995902
9,0,INS_OBS_LOAN_N,pearson_correlation_ge_0.90,PREV_APPROVED_N,0.962796


**Kết luận:** cả ba fold đều giữ 181 biến. Mỗi danh sách chỉ được học từ 123.004 hồ sơ thuộc phần fit, rồi áp dụng cho 61.502 hồ sơ giữ lại.


<a id="part-5"></a>

## 05. Fit trên toàn train và xuất tập biến

Fit lại cùng một bộ lọc trên toàn train để có danh sách phục vụ lần huấn luyện cuối. Số cột có thể khác giữa toàn train và các fold vì lượng dữ liệu dùng tính tương quan/độ thiếu khác nhau.

Danh sách này là đầu ra của quy tắc lọc đã chốt; quyết định về chất lượng và artifact cuối cần dựa vào đánh giá mô hình ở bước tiếp theo.

In [5]:
final_selected, final_dropped = select_filter_features(
    x, schema_b, correlation_limit=CORRELATION_LIMIT,
    min_pairwise_overlap=MIN_PAIRWISE_OVERLAP)
selected_schema = {
    'feature_order': final_selected,
    'numeric': [c for c in schema_b['numeric'] if c in final_selected],
    'categorical': [c for c in schema_b['categorical'] if c in final_selected]
}
(OUTPUT / 'final_selected_features_train_only.json').write_text(
    json.dumps(final_selected, indent=2), encoding='utf-8')
(OUTPUT / 'selected_schema_train_only.json').write_text(
    json.dumps(selected_schema, indent=2), encoding='utf-8')
pd.DataFrame(final_dropped).to_csv(
    OUTPUT / 'final_dropped_features_train_only.csv', index=False)

selected_table = pd.DataFrame({
    'feature': final_selected,
    'type': ['numeric' if c in schema_b['numeric'] else 'categorical'
             for c in final_selected]
})
selected_table.to_csv(OUTPUT / 'selected_features_train_only.csv', index=False)
frequency = pd.DataFrame({
    'feature': schema_b['feature_order'],
    'selected_folds': [
        sum(c in columns for columns in selected_by_fold.values())
        for c in schema_b['feature_order']],
    'selected_full_train': [
        c in final_selected for c in schema_b['feature_order']]
})
frequency.to_csv(OUTPUT / 'selection_frequency.csv', index=False)

display(pd.DataFrame([{
    'input_features': len(x.columns),
    'selected_features': len(final_selected),
    'numeric_selected': len(selected_schema['numeric']),
    'categorical_selected': len(selected_schema['categorical']),
    'removed_features': len(final_dropped)
}]))
display(pd.DataFrame(final_dropped))
display(selected_table.head(20))
display(frequency.loc[frequency['selected_folds'].between(1, 2)])


,input_features,selected_features,numeric_selected,categorical_selected,removed_features
0,199,182,166,16,17


,feature,reason,kept_instead,absolute_correlation
0,REGION_RATING_CLIENT_W_CITY,pearson_correlation_ge_0.90,REGION_RATING_CLIENT,0.951490
1,EMPLOYED_SENTINEL,pearson_correlation_ge_0.90,FLAG_EMP_PHONE,0.999816
2,BUR_CLOSED_N,pearson_correlation_ge_0.90,BUR_LOAN_N,0.933045
3,BB_CLOSED_MONTH_N,pearson_correlation_ge_0.90,BB_MONTH_N,0.907496
4,POS_LOAN_N,pearson_correlation_ge_0.90,PREV_APPROVED_N,0.951865
5,POS_KNOWN_DPD_MONTH_N,pearson_correlation_ge_0.90,POS_MONTH_N,1.000000
6,CC_KNOWN_DPD_MONTH_N,pearson_correlation_ge_0.90,CC_MONTH_N,1.000000
7,INS_OBS_VERSIONED_PERIOD_N,pearson_correlation_ge_0.90,INS_OBS_PAYMENT_ROW_N,0.995858
8,INS_OBS_LOAN_N,pearson_correlation_ge_0.90,PREV_APPROVED_N,0.962794
9,HAS_USABLE_CARD,pearson_correlation_ge_0.90,CC_LOAN_N,0.995333


,feature,type
0,NAME_CONTRACT_TYPE,categorical
1,CODE_GENDER,categorical
2,FLAG_OWN_CAR,categorical
3,FLAG_OWN_REALTY,categorical
4,CNT_CHILDREN,numeric
5,AMT_INCOME_TOTAL,numeric
6,AMT_CREDIT,numeric
7,AMT_ANNUITY,numeric
8,NAME_TYPE_SUITE,categorical
9,NAME_INCOME_TYPE,categorical


,feature,selected_folds,selected_full_train
16,FLAG_MOBIL,2,True
97,FLAG_DOCUMENT_10,2,True
99,FLAG_DOCUMENT_12,2,True


**Kết luận:** fit lại trên toàn train giữ 182 biến. Việc khác một biến so với từng fold xuất phát từ phạm vi dữ liệu tính tương quan và độ thiếu; đây là danh sách dành cho lần huấn luyện cuối.


<a id="part-6"></a>

## 06. Chốt kết quả và bàn giao bước tiếp theo

Bộ lọc trên toàn train giữ **182/199 biến**, gồm **166 biến số và 16 biến phân loại**. Có 17 biến bị loại theo quy tắc tương quan; không có cột hằng bị loại. Khi fit riêng trong từng fold, selector giữ **181 biến/fold**; danh sách giữa các fold có khác nhau.

Đầu ra được lưu trong `reports/modeling/feature_selection_20261028/`:

- `selected_features_by_fold.json`: danh sách riêng cho từng fold, dùng cho CV.
- `selected_schema_train_only.json`: schema 182 biến, dành cho lần fit cuối trên toàn train.
- `final_dropped_features_train_only.csv`: các biến bị loại và biến đại diện được giữ.
- `selection_frequency.csv`: mức ổn định của lựa chọn giữa các fold và toàn train.

Notebook 06 đã hoàn tất 63 lượt CV theo cùng quy tắc, dùng chung danh sách theo fold cho sáu họ mô hình. Phần đánh giá đó được lưu riêng trong `reports/modeling/feature_selection_redo_20261029/`; các danh sách biến khớp với đầu ra tại đây.

Feature Selection đã hoàn tất. Việc giảm số biến không tự bảo đảm hiệu năng tăng; kết quả CV được trình bày ở notebook 06. Notebook 07 đã hoàn tất fit bản cuối 182 biến, kiểm tra calibration và chọn ngưỡng; notebook 08 đã cập nhật phần giải thích và ví dụ chấm điểm.

[Quyết định chốt model 182 biến](../reports/modeling/selected_model_evaluation_20261030/project_model_decision.md).


In [6]:
protocol = {
    'method': 'constant-column filter, then Pearson redundancy filter',
    'scope': 'train features only; no labels loaded',
    'input_features': len(x.columns),
    'numeric_input': len(schema_b['numeric']),
    'categorical_input': len(schema_b['categorical']),
    'absolute_pearson_threshold': CORRELATION_LIMIT,
    'minimum_pairwise_overlap_fraction': MIN_PAIRWISE_OVERLAP,
    'representative_rule': 'lower missingness first, then original schema order',
    'categorical_policy': 'retained unless constant in the fit data',
    'cv_selection_scope': 'fit selector separately inside each training fold',
    'final_train_only_feature_count': len(final_selected),
    'numeric_selected': len(selected_schema['numeric']),
    'categorical_selected': len(selected_schema['categorical']),
    'final_train_only_dropped_count': len(final_dropped),
    'train_labels_read': False,
    'validation_labels_read': False,
    'test_labels_read': False,
    'model_evaluation': 'reported separately in notebook 06',
    'sources': [
        'Han et al., Data Mining: Concepts and Techniques, section 7.1',
        'Kele Xu, Data Mining Competition Practices, section 2.4'
    ]
}
(OUTPUT / 'protocol.json').write_text(
    json.dumps(protocol, indent=2), encoding='utf-8')
print('Feature Selection completed:', len(final_selected), 'features.')


Feature Selection completed: 182 features.
